In [0]:
CREATE WIDGET TEXT end_date DEFAULT '2025-12-31';

In [0]:
select distinct PATIENT_ID from com_raw.kom_medical_events where PROCEDURE_CODE = 'J1743' or NDC11 in ('54092070001','540920700')
union 
select distinct patient_id from com_edp_prd.com_raw.kom_pharmacy_events where ndc11 in ('54092070001','540920700')

In [0]:
-- CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary AS
WITH
-- -----------------------------
-- Eligibility (Specified + Incremental Unspecified)
-- -----------------------------
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
elaprase_patients as 
(select distinct PATIENT_ID from com_raw.kom_medical_events where PROCEDURE_CODE = 'J1743' or NDC11 in ('54092070001','540920700')
union 
select distinct patient_id from com_edp_prd.com_raw.kom_pharmacy_events where ndc11 in ('54092070001','540920700')
),

MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        -- SELECT DISTINCT PATIENT_ID
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE NDC11 IN ('54092070001','540920700')
        --   AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
        -- UNION ALL
        -- SELECT DISTINCT PATIENT_ID
        -- FROM com_edp_prd.com_raw.kom_pharmacy_events
        -- WHERE NDC11 IN ('54092070001','540920700')
        --   AND TRANSACTION_RESULT = 'PAID'
        --   AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
        -- UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
        --                          '38206','38230','38232','38240','38241','38242','38243','38250')
                                 WHERE PROCEDURE_CODE IN ('J1743') and PATIENT_ID not in (select distinct PATIENT_ID from elaprase_patients)
                                --  WHERE PROCEDURE_CODE IN ('J1743')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    ) t
),
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    ) t
),
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
)

SELECT * FROM eligible_patients;

In [0]:
select * from com_raw.kom_medical_events;

In [0]:
select * from com_raw.kom_pharmacy_events;

In [0]:
CREATE OR REPLACE TEMP VIEW runtime_parameters AS
 
SELECT
    (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events) AS max_medical_date,
 
    (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events) AS max_pharmacy_date,
 
    LAST_DAY(
        ADD_MONTHS(
            LEAST(
                (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events),
                (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events)
            ), -1
        )
    ) AS end_date,
 
    CURRENT_DATE() AS run_date;
 
    SELECT * FROM runtime_parameters;

In [0]:
WITH
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

    UNION

    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),

Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

    UNION

    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),

Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
eligible_dx_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified
    UNION
    SELECT PATIENT_ID FROM Patients_2Dx_Unspecified
),
-- select * from eligible_dx_patients;
elaprase_tx AS (
    SELECT DISTINCT PATIENT_ID
    FROM com_edp_prd.com_raw.kom_medical_events
     WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366', 'S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250','J1743') 
    --  WHERE PROCEDURE_CODE in ('J1743') 
     OR NDC11 IN ('54092070001','540920700')
     AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'

    UNION

    SELECT DISTINCT PATIENT_ID
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
)
-- SELECT DISTINCT PATIENT_ID
-- FROM elaprase_tx;
SELECT DISTINCT e.PATIENT_ID
FROM eligible_dx_patients e
INNER JOIN elaprase_tx t
ON e.PATIENT_ID = t.PATIENT_ID;

In [0]:
WITH
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

    UNION

    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),

Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

    UNION

    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),

Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (

        SELECT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE (
                NDC11 IN ('54092070001','540920700')
                OR PROCEDURE_CODE = 'J1743'
              )
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'

        UNION 

        SELECT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
    )
),

MPSII_Treatment_Other AS (
    SELECT DISTINCT PATIENT_ID
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366',
        'S9357','S9379',
        '38206','38230','38232','38240',
        '38241','38242','38243','38250'
    )
    AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
),

-- Patients_2Dx_Specified_With_Treatment AS (
--     SELECT DISTINCT p.PATIENT_ID
--     FROM Patients_2Dx_Specified p
--     INNER JOIN MPSII_Treatment_Other t
--     ON p.PATIENT_ID = t.PATIENT_ID
-- ),

-- Patients_Incremental_Unspecified AS (
--     SELECT DISTINCT p.PATIENT_ID
--     FROM Patients_2Dx_Unspecified p
--     INNER JOIN MPSII_Treatment_Elaprase_Only t
--     ON p.PATIENT_ID = t.PATIENT_ID
--     WHERE p.PATIENT_ID NOT IN (
--         SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
--     )
-- ),

eligible_patients AS (
    SELECT PATIENT_ID FROM mpsii_treatment_elaprase_only
    UNION
    SELECT PATIENT_ID FROM mpsii_treatment_other
)

SELECT DISTINCT PATIENT_ID
FROM eligible_patients 

###Yaman

In [0]:
-- CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary AS
WITH
-- -----------------------------
-- Eligibility (Specified + Incremental Unspecified)
-- -----------------------------
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-12-31'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-12-31'
),
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-12-31'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-12-31'
),
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
-- elaprase_patients as (select distinct PATIENT_ID from com_raw.kom_medical_events where PROCEDURE_CODE in ('J1743') or NDC11 in ('54092070001','540920700')
-- union 
-- select distinct patient_id from com_edp_prd.com_raw.kom_pharmacy_events where ndc11 in ('54092070001','540920700')),
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '2025-12-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
                                --  WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366', 'S9357','S9379',
                                --  '38206','38230','38232','38240','38241','38242','38243','38250') and PATIENT_ID not in (select distinct PATIENT_ID from elaprase_patients)
                                --  WHERE PROCEDURE_CODE IN ('J1743')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'
    ) t
),
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '2025-12-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'
    ) t
),
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
)
select *
from eligible_patients

In [0]:
-- CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary AS
WITH
-- -----------------------------
-- Eligibility (Specified + Incremental Unspecified)
-- -----------------------------
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-12-31'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-12-31'
),
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-12-31'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-12-31'
),
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2023-08-01' AND '2025-12-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE IN ('J1743')
        -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250','J1743') 
        AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'
    ) t
),
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2023-08-01' AND '2025-12-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
        AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'
    ) t
),
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN mpsii_treatment_elaprase_only t USING (PATIENT_ID)
),
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
)
select *
from eligible_patients

In [0]:
select b.ndc11, count(distinct a.patient_id) from cmpa_insights_internal_schema.patient360_master a
left join com_intgr.claims_pharmacy_events b
on a.patient_id = b.PATIENT_ID
where b.NDC11 in ('50383066730','69097022416')
group by 1
order by 2 desc

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.reference_file;